In [1]:
%pip install -U torch transformers accelerate huggingface-hub pandas pyarrow "chronos-forecasting>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 39.7 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 26.7 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 28.2 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 37.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 43.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 44.3 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 45.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 32.2 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 47.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 43.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [1]:
import os
import shutil
import torch

total, used, free = shutil.disk_usage(".")

print("Python environment:", os.sys.executable)
print(f"Free disk: {free / 1024**3:.2f} GB")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )
else:
    print("No GPU detected; models will run on CPU.")

Python environment: /home/coder/agentic-forecasting/.venv/bin/python
Free disk: 17.87 GB
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: Tesla T4
GPU memory: 14.58 GB


In [12]:
from huggingface_hub import model_info

model_ids = [
    "google/timesfm-2.5-200m-transformers",
    "amazon/chronos-2",
]

for model_id in model_ids:
    try:
        info = model_info(model_id)
        print(f"{model_id} is accessible")
        print("   Revision:", info.sha[:8])
    except Exception as error:
        print(f" Cannot access {model_id}")
        print("  ", type(error).__name__, error)

google/timesfm-2.5-200m-transformers is accessible
   Revision: 5a9806b9
amazon/chronos-2 is accessible
   Revision: 29ec3766


### Test TimesFM

In [13]:
import torch
from transformers import TimesFm2_5ModelForPrediction

timesfm = TimesFm2_5ModelForPrediction.from_pretrained(
    "google/timesfm-2.5-200m-transformers"
).to("cuda", dtype=torch.float32).eval()

# 64 is divisible by TimesFM's patch length of 32
past_values = [
    torch.linspace(100.0, 150.0, 64, device="cuda")
]

with torch.no_grad():
    timesfm_result = timesfm(
        past_values=past_values,
        forecast_context_len=64,
    )

print("TimesFM ok")
print("Forecast shape:", timesfm_result.mean_predictions.shape)
print("First six predictions:")
print(timesfm_result.mean_predictions[0, :6].cpu())

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

TimesFM ok
Forecast shape: torch.Size([1, 128])
First six predictions:
tensor([151.1108, 151.4288, 151.8981, 153.3083, 153.9243, 155.0537])


### Test Chrono

In [14]:
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline

history = pd.DataFrame({
    "item_id": ["deposits"] * 60,
    "timestamp": pd.date_range("2021-01-01", periods=60, freq="MS"),
    "target": (
        100
        + np.arange(60) * 0.5
        + 5 * np.sin(np.arange(60) / 3)
    ),
})

print("Loading Chronos-2...")

chronos = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda",
)

chronos_result = chronos.predict_df(
    history,
    prediction_length=6,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id",
    timestamp_column="timestamp",
    target="target",
)

print("Chronos-2 ok")
chronos_result

Loading Chronos-2...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 ok


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,deposits,2026-01-01,target,134.675491,133.610687,134.675491,135.781250
1,deposits,2026-02-01,target,135.432373,134.429245,135.432373,136.498398
2,deposits,2026-03-01,target,135.733917,134.726654,135.733917,136.762558
3,deposits,2026-04-01,target,135.295364,134.436462,135.295364,136.551041
4,deposits,2026-05-01,target,134.430496,133.285507,134.430496,135.545059
5,deposits,2026-06-01,target,133.431534,132.297256,133.431534,134.563919


In [ ]:
### Clear cache
import gc

del timesfm
del timesfm_result
gc.collect()
torch.cuda.empty_cache()

print("TimesFM cleared from GPU memory")